We estimate the value of the tuning parameter $\zeta$ that gives a correlation between the in- and out-degrees of $0.55$, which is consistent with empirical findings. 

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "network_generator").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import torch
import torch.nn as nn
from network_generator.config import NetworkConfig, TargetConfig
from network_generator.initialization import (
    directed_configuration_model,
    generate_burr_degrees,
    generate_burr_degrees_matching_sum,
    generate_initial_strength_fitnesses,
    get_parameters_for_burr_distribution,
    triangular_wedge_matching,
)
from network_generator.metrics import (
    compute_degrees_from_model,
    compute_hill_exponent,
)

criterion_sse = nn.MSELoss(reduction="sum")
criterion_mse = nn.MSELoss()

In [ ]:
# Configuration - modify these config classes to change parameters
network_config = NetworkConfig(num_firms=100_000)
target_config = TargetConfig()

# For backward compatibility, extract values to individual variables
num_firms = network_config.num_firms
monetary_unit = network_config.monetary_unit
country_code = network_config.country_code
country_year = network_config.country_year

In [ ]:
# Extract target values from config for backward compatibility
target_hill_in_deg = torch.tensor(target_config.target_hill_in_deg, dtype=torch.float32)
target_hill_out_deg = torch.tensor(target_config.target_hill_out_deg, dtype=torch.float32)
target_hill_weight_distribution = torch.tensor(target_config.target_hill_weight_distribution, dtype=torch.float32)
target_hill_in_str = torch.tensor(target_config.target_hill_in_str, dtype=torch.float32)
target_hill_out_str = torch.tensor(target_config.target_hill_out_str, dtype=torch.float32)
target_average_connection = torch.tensor(target_config.target_average_connection, dtype=torch.float32)

target_so_ko_corr = torch.tensor(target_config.target_so_ko_corr, dtype=torch.float32)
target_si_ki_corr = torch.tensor(target_config.target_si_ki_corr, dtype=torch.float32)
target_si_so_corr = torch.tensor(target_config.target_si_so_corr, dtype=torch.float32)
target_ki_ko_corr = torch.tensor(target_config.target_ki_ko_corr, dtype=torch.float32)

target_so_ko_regcoeff = torch.tensor(target_config.target_so_ko_regcoeff, dtype=torch.float32)
target_ko_so_regcoeff = torch.tensor(target_config.target_ko_so_regcoeff, dtype=torch.float32)
target_si_ki_regcoeff = torch.tensor(target_config.target_si_ki_regcoeff, dtype=torch.float32)
target_ki_si_regcoeff = torch.tensor(target_config.target_ki_si_regcoeff, dtype=torch.float32)

target_tls_str = torch.tensor(target_config.target_tls_str, dtype=torch.float32)
target_tls_deg = torch.tensor(target_config.target_tls_deg, dtype=torch.float32)

In [ ]:
# Extract Burr distribution parameters from network_config
burr_max_indeg = torch.tensor(network_config.burr_max_indeg, dtype=torch.float32)
burr_max_outdeg = torch.tensor(network_config.burr_max_outdeg, dtype=torch.float32)

burr_alpha_outdeg = network_config.burr_alpha_outdeg
burr_var_log_outdeg = network_config.burr_var_log_outdeg

burr_alpha_indeg = network_config.burr_alpha_indeg
burr_var_log_indeg = network_config.burr_var_log_indeg

In [ ]:
from tqdm import tqdm
import numpy as np

correlation_dict = {}

for tuning_val in tqdm([-0.20,-0.19,-0.18,-0.17,-0.16,-0.15,-0.14,-0.13]):
    correlation_list = []
    for _ in range(50):
        k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(burr_alpha_outdeg, burr_var_log_outdeg, target_average_connection)
        k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(burr_alpha_indeg, burr_var_log_indeg, target_average_connection)
        out_deg_seq_burr = generate_burr_degrees(
            c_val=c_outdeg,
            d_val=k_outdeg,
            lam_val=lam_outdeg,
            max_cutoff=burr_max_outdeg,
            num_of_firms=num_firms,
        )
        itr = 0
        while (
            (out_deg_seq_burr.max().item() < 0.1 * num_firms)
            or (out_deg_seq_burr[out_deg_seq_burr > 0].log().var().item() < 2.95)
            or (
                compute_hill_exponent(
                    out_deg_seq_burr,
                    network_config.threshold_percentile_for_hill_deg,
                    is_degree=True,
                ).item()
                > 1.5
            )
            or (
                compute_hill_exponent(
                    out_deg_seq_burr,
                    network_config.threshold_percentile_for_hill_deg,
                    is_degree=True,
                ).item()
                < 1.4
            )
            or (out_deg_seq_burr.float().mean() < 40.0)
        ):
            out_deg_seq_burr = generate_burr_degrees(
                c_val=c_outdeg,
                d_val=k_outdeg,
                lam_val=lam_outdeg,
                max_cutoff=burr_max_outdeg,
                num_of_firms=num_firms,
            )
            itr += 1
            
        in_deg_seq_burr = generate_burr_degrees_matching_sum(
            c_val=c_indeg,
            d_val=k_indeg,
            lam_val=lam_indeg,
            max_cutoff=burr_max_indeg,
            num_of_firms=num_firms,
            degree_sum=out_deg_seq_burr.sum(),
        )
        while (
            (in_deg_seq_burr.min().item() < 0)
            or (in_deg_seq_burr[in_deg_seq_burr > 0].log().var().item() < 1.95)
            or (
                compute_hill_exponent(
                    in_deg_seq_burr,
                    network_config.threshold_percentile_for_hill_deg,
                    is_degree=True,
                ).item()
                < 2.3
            )
            or (
            compute_hill_exponent(
                in_deg_seq_burr,
                network_config.threshold_percentile_for_hill_deg,
                is_degree=True,
            ).item()
            > 2.4
            )
        ):
            in_deg_seq_burr = generate_burr_degrees_matching_sum(
                c_val=c_indeg,
                d_val=k_indeg,
                lam_val=lam_indeg,
                max_cutoff=burr_max_indeg,
                num_of_firms=num_firms,
                degree_sum=out_deg_seq_burr.sum(),
            )
            itr += 1
            
        test_reordered_out_deg_seq_burr = triangular_wedge_matching(
            in_deg_seq=in_deg_seq_burr,
            out_deg_seq=out_deg_seq_burr,
            noise_strength=tuning_val,
        )

        adj_mat_init = directed_configuration_model(
            in_deg_seq_burr, test_reordered_out_deg_seq_burr
        )
        in_deg_seq_init, out_deg_seq_init = compute_degrees_from_model(adj_mat_init)

        in_str_seq_init, out_str_seq_init = generate_initial_strength_fitnesses(
            mean_in_str=network_config.mean_for_gen_lognorm_fi,
            mean_out_str=network_config.mean_for_gen_lognorm_fo,
            var_in_str=in_deg_seq_init[in_deg_seq_init > 0].log().var().item(),
            var_out_str=out_deg_seq_init[out_deg_seq_init > 0].log().var().item(),
            correlation=network_config.corr_lognorm_gen_fi_fo,
            num_of_firms=num_firms,
            monetary_unit=monetary_unit,
        )

        ki_ko_corr = torch.corrcoef(
            torch.stack(
                (in_deg_seq_init[(in_deg_seq_init > 0)&(out_deg_seq_init>0)&(in_str_seq_init>0)&(out_str_seq_init>0)].log(), 
                out_deg_seq_init[(in_deg_seq_init > 0)&(out_deg_seq_init>0)&(in_str_seq_init>0)&(out_str_seq_init>0)].log(), 
                )
                )
            )[0, 1]
        correlation_list.append(ki_ko_corr.item())
    correlation_list = np.array(correlation_list)
    correlation_dict[tuning_val] = (correlation_list.mean(), correlation_list.std(), correlation_list.max(), correlation_list.min())

In [ ]:
import pandas as pd

# Convert dictionary to DataFrame
df = pd.DataFrame.from_dict(
    correlation_dict,
    orient='index',
    columns=['mean', 'std', 'max', 'min']
)

# Optional: make tuning_val a column instead of index
df = df.reset_index().rename(columns={'index': 'tuning_val'})

# Save to CSV
df.to_csv('./paper/artifacts/tuning_csvs/zeta_reordering_correlation_sweep.csv', index=False)